In [ ]:
## The speculative decoding algorithm
    ## Two models - P (we're trying to speed up) and Q (a much smaller approximation of P, preferably a smaller version of the 
    ## same family)
    ## Sample from Q gamma times and verify using P for each generation parallely. 
        ## (Q) What does verifying look like? Do we perform forward pass on each token and check the probability distribution? 
        ## (A) YES 
        
        ## (Q) But why is this faster? 
        ## (A) 
    
    ## Here's how it goes, according to how I understand it: 
        ## (I) We have the following: (1) A base model P that we want to speed up (2) A draft model Q that is faster and
        ## (3) A prefix/input sequence that has M tokens
        ## (II) For any given prefix/input sequence, we generate N new tokens using the draft model and generate a new sequence 
        ## with 
        ## (III) Now, P will verify the N new tokens using the following algorithm: 
            ## Compute the softmax after doing a forward pass
            ## Check the probability ratio
                ## if it's above certain threshold it's good to go, 
                ## Else, adjust P's probabilities

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

TARGET_MODEL = "Qwen2.5-7B-Instruct"
DRAFT_MODEL  = "Qwen2.5-0.5B-Instruct-1M"

model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
prompt = "Give me a short introduction to large language model."

def base_generate(target_model, target_tokenizer, prompt):    
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors='pt').to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=20
    )
    
    generated_ids = [
        output_ids[len(input_ids): ] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

@torch.no_grad()
def draft_generate(draft_model, draft_tokenizer, input_ids, num_tokens):
    draft_tokens = []
    dragt_probs = []
    
    current_ids = input_ids.clone()
    
    for _ in range(num_tokens):
        outputs = draft_model(current_ids)
        logits = outputs.logits[0, -1, :]
        
        probs = torch.softmax(logits, dim=0)
        
        next_token = torch.multinomial(probs, num_samples=1)
        token_id = next_token.item()
        
        draft_tokens.append(token_id)
        draft_probs.append(probs[token_id].item())
        
        torch.cat([current_ids, next_token.unsqueeze(0)], dim=1)
    
    return draft_tokens, draft_probs

def target_verify():
    pass